# 03b — ResNet1D

The third model in spec section I: *"ResNet1D / lightweight
Transformer — stronger deep baseline reported in recent
PulseDB-benchmarking papers"*.

Same convolutional idea as the CNN, with skip connections so a much
deeper stack still trains. This is the standard strong baseline
across the PulseDB literature.

About 6 hours on a T4 at 50 epochs.

---

**This notebook holds no pipeline code.** Everything lives in
`src/train.py`, shared by all five models and all four protocols, so the
difference between this notebook and the other two really is only the
architecture — not a batch size, not a split, not a scaling choice. The
whole comparison depends on that.

To change how training works, edit `src/train.py` and re-upload `src/`
to Drive (0.1 MB). The data does not need re-uploading.

## Setup

Run these cells in order before anything else. They work both on Colab
and on a laptop — on Colab the data is copied off Drive onto the local
disk first, because Drive is a network mount and the sampler reads clips
in random order, which would leave the GPU waiting on I/O. The copy also
stops immediately if the upload turned out to be incomplete.

In [ ]:
# Needs a GPU. Runtime > Change runtime type > T4 GPU.
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU'

In [ ]:
import os, sys, glob, shutil, time

IN_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content/drive')

if not IN_COLAB:
    # running from notebooks/ on a normal machine
    sys.path.insert(0, '../src')
    sys.path.insert(0, '../scripts')
else:
    from google.colab import drive
    drive.mount('/content/drive')

    # Find the bp folder wherever it was dropped in Drive.
    hits = glob.glob('/content/drive/MyDrive/**/data/colab_manifest.json',
                     recursive=True)
    if len(hits) != 1:
        raise SystemExit(f'expected exactly one bp folder, found {len(hits)}:'
                         + ''.join('\n  ' + h for h in hits))
    DRIVE_ROOT = os.path.dirname(os.path.dirname(hits[0]))
    LOCAL_ROOT = '/content/bp'
    print('drive:', DRIVE_ROOT)

    # Copy the data to local disk first. Drive is a network mount and the
    # sampler reads clips in random order, so training straight off Drive
    # leaves the GPU idle waiting on I/O. A few minutes here saves hours.
    t0 = time.time()
    for rel in ['src', 'scripts', 'data/colab_manifest.json', 'data/features',
                'data/processed/pulsedb', 'data/processed/ppg_dalia/windows']:
        src, dst = f'{DRIVE_ROOT}/{rel}', f'{LOCAL_ROOT}/{rel}'
        if os.path.exists(dst):
            continue
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        (shutil.copytree if os.path.isdir(src) else shutil.copy2)(src, dst)
        print(f'  copied {rel:40s} {time.time() - t0:5.0f}s')

    # results/ points back at Drive — metrics, predictions and checkpoints
    # all land under results/models/<name>/, so a dropped session does not
    # take the run with it.
    os.makedirs(f'{DRIVE_ROOT}/results/models', exist_ok=True)
    if not os.path.islink(f'{LOCAL_ROOT}/results'):
        os.symlink(f'{DRIVE_ROOT}/results', f'{LOCAL_ROOT}/results')

    os.environ['BP_ROOT'] = LOCAL_ROOT     # every path follows this
    sys.path.insert(0, LOCAL_ROOT + '/src')
    sys.path.insert(0, LOCAL_ROOT + '/scripts')

    # Fail now if the upload was incomplete, not twenty minutes into a run.
    from prepare_colab_upload import verify
    if verify(LOCAL_ROOT):
        raise SystemExit('copy is incomplete — see the list above')
    print(f'\nBP_ROOT = {LOCAL_ROOT}   (ready in {time.time() - t0:.0f}s)')

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU — set the runtime type and restart')

## Training

`calfree` first: 144 patients the model has never seen, and the number
that goes in the write-up. If the session drops later, the result that
matters is already saved.

Each run writes its metrics and its predictions to Drive, so the other
protocols can be run on another day without repeating this one.

In [ ]:
from train import run

MODEL  = 'resnet'
EPOCHS = 50
BATCH  = 256

calfree = run(model=MODEL, protocol='calfree',
              epochs=EPOCHS, batch_size=BATCH)

## The comparability run

One extra run: `calfree` with the BP-bin balancing switched off.

Balancing is a spec requirement (section L) and it is on by default, but
it changes the headline MAE — it trades average error for accuracy on the
hypertensive patients, who are rare. Every published PulseDB number was
measured without it, and section K makes comparability the reason MAE is
the primary metric. So both are reported: the balanced number is this
project's model, the unbalanced one is what lines up against the papers.

In [ ]:
calfree_nobalance = run(model=MODEL, protocol='calfree', balance=False,
                        tag='nobalance',
                        epochs=EPOCHS, batch_size=BATCH)

## Seed variance

Two more runs each of `calfree` and `calfree_nobalance`, seeds 1 and 2 —
three total per protocol with the runs above (seed 0). A single training
run is a point estimate; the headline claims (this project vs. DMT, and
the personalization result built on top of the balanced run) only mean
something if they survive re-training. Saved with a `seed1`/`seed2` tag,
so they sit beside the canonical runs rather than overwriting them.

In [ ]:
seeds = {}
for seed in (1, 2):
    seeds[('calfree', seed)] = run(model=MODEL, protocol='calfree', seed=seed,
                                    tag=f'seed{seed}', epochs=EPOCHS, batch_size=BATCH)
    seeds[('nobalance', seed)] = run(model=MODEL, protocol='calfree', seed=seed,
                                      balance=False, tag=f'nobalance_seed{seed}',
                                      epochs=EPOCHS, batch_size=BATCH)

## The other three protocols

Same architecture, same settings, same training clips. Only the test
patients change.

This cell trains three more models and takes roughly three times as long
as the one above — run it once you are happy with the `calfree` result.

In [ ]:
others = {}
for proto in ('calbased', 'aami', 'leaky'):
    others[proto] = run(model=MODEL, protocol=proto,
                        epochs=EPOCHS, batch_size=BATCH)

### Seed variance, on calbased and leaky too

The professor's follow-up: the architecture-dependent leakage-gap ratio (calfree vs. calbased/leaky) is only as trustworthy as both sides of it — calfree already gets 3 seeds above, this does the same for calbased and leaky. `aami` is left at one seed; it isn't part of the leakage-gap ratio.

In [ ]:
proto_seeds = {}
for proto in ('calbased', 'leaky'):
    for seed in (1, 2):
        proto_seeds[(proto, seed)] = run(model=MODEL, protocol=proto, seed=seed,
                                          tag=f'seed{seed}', epochs=EPOCHS, batch_size=BATCH)

## The domain-shift test

Spec sections E, H and L: the model is trained on PulseDB and then run
on PPG-DaLiA, a wrist-worn free-living dataset it has never seen. That
external test is the design choice the spec calls *"the most novel ...
differentiating this from typical single-dataset cuffless-BP papers"*.

PPG-DaLiA carries no BP labels, so there is no error to report. What is
measured instead: whether the answers stay physiologically possible,
how far the whole prediction distribution moves, whether the model can
still tell the fifteen volunteers apart, and whether it degrades with
motion.

This reuses the checkpoint saved by the `calfree` run above rather than
training again — the same model has to score both sides.

In [ ]:
from train import evaluate_dalia

# restated here so this cell runs on its own after a session restart,
# without re-running the training cells above
MODEL = 'resnet'

shift = evaluate_dalia(model=MODEL, protocol='calfree')

### The control

The same test on the unbalanced model, and it is not optional. BP-bin
balancing pushes a model to spread its predictions apart; that spread
is what appears to collapse on PPG-DaLiA, and part of the distribution
shift comes from it too. On gradient boosting the control cut the
measured shift from +32 to +19 mmHg and removed the collapse entirely.

Without this run there is no way to say how much of the domain shift is
the domain and how much is the weighting.

In [ ]:
shift_nobalance = evaluate_dalia(model=MODEL, protocol='calfree',
                                 tag='nobalance')

### Seed variance, on PPG-DaLiA too

The same domain-shift check, on the seed 1 and seed 2 checkpoints from
above — inference only, no training, seconds per run. Confirms whether
the shift numbers are as seed-stable as the calfree MAE.

In [ ]:
dalia_seeds = {}
for seed in (1, 2):
    dalia_seeds[('calfree', seed)] = evaluate_dalia(
        model=MODEL, protocol='calfree', seed=seed, tag=f'seed{seed}')
    dalia_seeds[('nobalance', seed)] = evaluate_dalia(
        model=MODEL, protocol='calfree', seed=seed, tag=f'nobalance_seed{seed}')

## 3-seed ensemble

Averaging the three seeds' predictions is free accuracy — no extra training,
just three numbers averaged per clip — and it almost always beats any single
seed. Reads straight from the saved prediction files, so this cell also works
on its own after a session restart.

In [ ]:
import numpy as np
from paths import model_dir

def _load(tag=''):
    p = model_dir(MODEL) / f"{MODEL}_calfree{'_' + tag if tag else ''}_predictions.npz"
    d = np.load(p)
    return d['y_true'], d['y_pred']

y_true, p0 = _load()
_, p1 = _load('seed1')
_, p2 = _load('seed2')
ensemble = (p0 + p1 + p2) / 3
for t, name in enumerate(('sbp', 'dbp')):
    single = np.abs(p0[:, t] - y_true[:, t]).mean()
    ens = np.abs(ensemble[:, t] - y_true[:, t]).mean()
    print(f'{name.upper()}  seed0 alone {single:.2f}  ->  3-seed ensemble {ens:.2f}')

## Results

In [ ]:
import pandas as pd

rows = []
runs = {'calfree': calfree, 'calfree (no balancing)': calfree_nobalance,
        **others}
for proto, out in runs.items():
    for target in ('sbp', 'dbp'):
        m = out['results'][target]
        rows.append(dict(protocol=proto, target=target, mae=m['mae'],
                         sde=m['sde'], r=m['r'],
                         aami='PASS' if m['aami_pass'] else 'FAIL',
                         bhs=m['bhs']))
pd.DataFrame(rows).round(2)

In [ ]:
# Everything this session produced, on Drive
from paths import MODEL_RESULTS
for p in sorted(MODEL_RESULTS.glob('*/*.json')):
    print(f'{p.stat().st_size / 1e3:8.1f} kB  {p.name}')